In [1]:
import pandas as pd
import os
from tqdm import tqdm
import json
from itertools import product

DO_ALL_MEETINGS = True
AMI_PATH = '/home/sgoyal/Projects/llm_asr_clarification/shared/datasets/amicorpus/validation'
# AMI_PATH = '/home/pkongsomjit/Projects/llm_asr_clarification/shared/datasets/amicorpus/validation'
# MEETING_TO_DO= '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/ES2005d'
QUESTION_FILE = 'parsed_diarized_gt'

# Which quiz results to load
# QUIZ_FILE = "vllm_quiz.json.bak_20261104"  # backup taken before the current job
# QUIZ_FILE = "vllm_quiz.json.bak_20261022"  # older backup
QUIZ_FILE = "vllm_quiz.json"  # latest results

ARGUMENTS = {
    'splits' : ['validation'],
    'clarification_num_lines': ["10", "20", "30", "40", "50"],
    'importance_detectors' : ["LSTM", "GT", "ALL"],
    'mistranscript_detectors' : ["RF", "GT", "ALL"]
}

keys = ARGUMENTS.keys()
values = ARGUMENTS.values()

# 2. Compute the Cartesian product and rebuild dictionaries
combinations = [dict(zip(keys, v)) for v in product(*values)]

TRANSCRIPT_FILES = ['parsed_diarized_gt', 'custom_transcript_gt_segments']
for arg in combinations:

    split = arg['splits']
    clarification_num_line = arg['clarification_num_lines']
    importance_detector = arg['importance_detectors']
    mistranscript_detector = arg['mistranscript_detectors']

    TRANSCRIPT_FILES.append(f"custom_transcript_gt_segments_{mistranscript_detector.lower()}_{importance_detector.lower()}_{clarification_num_line}_clarify4")

TRANSCRIPT_FILES = [f'score_using_{t}' for t in TRANSCRIPT_FILES]

# directories of meetings
if DO_ALL_MEETINGS:
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]

list_of_quiz_dicts = []
for meeting_path in tqdm(meeting_paths):
    # question_path = os.path.join(meeting_path, "quiz", f"quiz_from_{QUESTION_FILE}.json")
    # question_path = os.path.join(meeting_path, "quiz", "old_quiz.json")
    question_path = os.path.join(meeting_path, "quiz", QUIZ_FILE)
        
    # chatgpt = OpenAIWrapper()
    
    # Read question
    try:
        with open(question_path, "r", encoding="utf-8") as f:
            quiz = f.read()

        quiz = json.loads(quiz)
        meeting_name = meeting_path.split("/")[-1]
        for q in quiz:
            q['meeting_name'] = meeting_name
        list_of_quiz_dicts += quiz
    except Exception as err:
        print(f"couldnt open file {question_path}")

df = pd.DataFrame(list_of_quiz_dicts)

100%|██████████| 18/18 [00:00<00:00, 608.02it/s]


In [2]:
df.head()

,question,correct_answer,answer_using_parsed_diarized_gt,score_using_parsed_diarized_gt,answer_using_custom_transcript_gt_segments_rf_lstm_10_clarify4,score_using_custom_transcript_gt_segments_rf_lstm_10_clarify4,answer_using_custom_transcript_gt_segments_gt_lstm_10_clarify4,answer_using_custom_transcript_gt_segments,score_using_custom_transcript_gt_segments,score_using_custom_transcript_gt_segments_gt_lstm_10_clarify4,...,score_using_custom_transcript_gt_segments_all_oraclemask_30_clarify4,score_using_custom_transcript_gt_segments_all_oracleswap_40_clarify4,score_using_custom_transcript_gt_segments_all_oracleswapstd_40_clarify4,score_using_custom_transcript_gt_segments_all_oraclemask_40_clarify4,score_using_custom_transcript_gt_segments_all_oracleswap_50_clarify4,score_using_custom_transcript_gt_segments_all_oracleswapstd_50_clarify4,score_using_custom_transcript_gt_segments_all_oraclemask_50_clarify4,answer_using_custom_transcript_gt_segments_all_oracleattr_50_clarify4,score_using_custom_transcript_gt_segments_all_oracleattr_50_clarify4,meeting_name
0,What was the main reason the team decided to c...,Because the fruit-shaped versions were awkward...,The main reason the team decided to change the...,1,The main reason the team decided to change the...,1,The main reason the team decided to change the...,The main reason the team decided to change the...,1,1,...,1,1,1,1,1,1,1,The main reason the team decided to change the...,1,ES2011d
1,What did the team conclude about the relations...,A round or bulky fruit shape made the remote l...,The team concluded that the remote's shape sig...,0,The team concluded that the remote's shape sig...,0,The team concluded that the remote's shape sig...,The team concluded that the remote's shape sig...,0,0,...,0,1,1,0,1,1,1,The team concluded that the remote's shape sig...,0,ES2011d
2,What was the key issue with the evaluation sca...,They had reversed the scale and initially thou...,The group realized that they had misunderstood...,1,The key issue with the evaluation scale was co...,0,The key issue with the evaluation scale was co...,The key issue with the evaluation scale was co...,0,0,...,1,1,1,1,0,1,1,The key issue with the evaluation scale was a ...,0,ES2011d
3,According to the corrected evaluation criteria...,1 meant true and 7 meant false.,According to the corrected evaluation criteria...,1,"According to the meeting transcript, the numbe...",1,The number 1 represented something that is fan...,"According to the meeting transcript, the numbe...",0,0,...,1,1,1,0,0,1,0,"According to the meeting transcript, the numbe...",1,ES2011d
4,Which evaluation criterion was identified as t...,"Material being technologically innovative, spe...",Material technologically innovative (3.1),1,Material technologically innovative (3.1),1,3.1,3.1,1,1,...,1,1,1,1,1,1,1,Material technologically innovative (3.1),1,ES2011d


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Columns: 223 entries, question to meeting_name
dtypes: int64(110), str(113)
memory usage: 313.7 KB


In [4]:
for transcript_file in TRANSCRIPT_FILES:
    df[transcript_file] = pd.to_numeric(df[transcript_file], errors="coerce")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Columns: 223 entries, question to meeting_name
dtypes: int64(110), str(113)
memory usage: 313.7 KB


In [5]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    print(df[transcript_file].mean())

score_using_parsed_diarized_gt
0.95
score_using_custom_transcript_gt_segments
0.8555555555555555
score_using_custom_transcript_gt_segments_rf_lstm_10_clarify4
0.8333333333333334
score_using_custom_transcript_gt_segments_gt_lstm_10_clarify4
0.8444444444444444
score_using_custom_transcript_gt_segments_all_lstm_10_clarify4
0.8611111111111112
score_using_custom_transcript_gt_segments_rf_gt_10_clarify4
0.8722222222222222
score_using_custom_transcript_gt_segments_gt_gt_10_clarify4
0.8833333333333333
score_using_custom_transcript_gt_segments_all_gt_10_clarify4
0.8833333333333333
score_using_custom_transcript_gt_segments_rf_all_10_clarify4
0.8444444444444444
score_using_custom_transcript_gt_segments_gt_all_10_clarify4
0.8444444444444444
score_using_custom_transcript_gt_segments_all_all_10_clarify4
0.8611111111111112
score_using_custom_transcript_gt_segments_rf_lstm_20_clarify4
0.85
score_using_custom_transcript_gt_segments_gt_lstm_20_clarify4
0.8555555555555555
score_using_custom_transcript_gt

In [6]:
import re

# pattern = r"^score_using_custom_transcript_gt_segments_(rf|gt|all|random)_(oracleswapstd|oracleswap|oraclemask|lstmmask|lstm|gt|all)_(10|20|30|40|50)(_mul)?_clarify4$"

pattern = r"^score_using_custom_transcript_gt_segments_(rf|gt|all)_(oracleattr|oraclemask|lstmmask|lstm|gt|all)_(10|20|30|40|50)(_mul)?_clarify4$"

results = {}
for col in df.columns:
    match = re.match(pattern, col)
    # Skip columns that don't have a score for every question yet
    if match and df[col].notna().all():
        mis, imp, n, mul = match.groups()
        name = f"{mis.upper()} + {imp.upper()}" + (" (multiply)" if mul else "")
        results.setdefault(name, {})[n] = df[col].mean()

table = pd.DataFrame(results).T[["10", "20", "30", "40", "50"]].sort_index()
print(table.round(3).to_string())

print("\nNo clarification:", round(df["score_using_custom_transcript_gt_segments"].mean(), 3))
print("Perfect transcript:", round(df["score_using_parsed_diarized_gt"].mean(), 3))

                             10     20     30     40     50
ALL + ALL                 0.861  0.861  0.844  0.872  0.833
ALL + GT                  0.883  0.883  0.906  0.900  0.906
ALL + LSTM                0.861  0.833  0.844  0.856  0.867
ALL + LSTMMASK            0.856  0.844  0.850  0.861  0.878
ALL + ORACLEATTR            NaN    NaN    NaN    NaN  0.894
ALL + ORACLEMASK          0.872  0.889  0.900  0.917  0.922
GT + ALL                  0.844  0.856  0.850  0.850  0.861
GT + GT                   0.883  0.878  0.878  0.867  0.878
GT + LSTM                 0.844  0.856  0.828  0.844  0.850
RF + ALL                  0.844  0.844  0.833  0.844  0.828
RF + GT                   0.872  0.872  0.878  0.878  0.878
RF + LSTM                 0.833  0.850  0.844  0.861  0.856
RF + LSTMMASK             0.850  0.828  0.844  0.872  0.856
RF + LSTMMASK (multiply)  0.833  0.839  0.861  0.861  0.856

No clarification: 0.856
Perfect transcript: 0.95
